# 📱 Playground Series S6E8 - Predicting Smartphone Addiction / `S6E8: 🚀🔥0.97123`

| 項目 | 内容 |
|---|---|
| コンペ | [Predicting Smartphone Addiction (Playground S6E8)](https://www.kaggle.com/competitions/playground-series-s6e8) |
| 元notebook | [S6E8: 🚀🔥0.97123](https://www.kaggle.com/code/itzzomkar/s6e8-0-97123) |
| 原著者 | OMKAR KADAM |
| Public Score | **0.97123** |
| ライセンス | Apache 2.0 |
| 実行時間 | 23秒（モデル学習なし） |

## 手法の概要

**モデルを1つも学習しません。** 公開されている上位notebookの `submission.csv` を5本ほど読み込み、**Rank-Gauss変換をかけてから重み付き平均する**だけの、わずか60行程度のブレンドnotebookです。それで0.97123 — この時点の公開最上位帯に並びます。

やっていることは3ステップ。

1. `/kaggle/input/` 以下を再帰的に探して、`submission.csv` を**動的に全部集める**。
2. 各予測を **順位 → 一様分布 → 正規分布** に変換する（Rank-Gauss / rank-based inverse normal transform）。
3. ファイル名に `elite` / `public` / `psa` / `would` / `ensemble` を含むものに重み0.35、それ以外に0.15を与えて加重平均し、最後にまた順位に戻して提出。

> ⚠️ **これは学習目的の解説付き写しです。** コード本体は原著のまま変更していません（出力は含めず未実行）。実行にはKaggle上で他notebookの出力をInputとして添付する必要があります。
>
> 📌 **この写しをあえて選んだ理由**: 短さのわりに、**「AUCという指標の性質を利用して、学習ゼロでスコアを買う」** という発想が非常にきれいに現れているためです。同時に、後述するとおり**このやり方の危うさ**も詰まっており、教材として両面から使えます。

## 📏 評価指標と、それがこの手法を成立させている理由

### タスクと指標
`addicted_label`（スマートフォン依存かどうか）を予測する**二値分類**で、指標は **ROC-AUC** です。

AUCは「無作為に選んだ正例1件と負例1件について、正例のスコアのほうが高い確率」に等しい、と解釈できます。ここから決定的な性質が導かれます。

> **AUCは予測値の「順序」だけで決まり、「値そのもの」には一切依存しない。**

予測を `p` から `p²` にしても、`log(p)` にしても、順位が保たれる限りAUCは1ビットも変わりません。

### なぜこの性質がブレンドを強力にするのか

1. **キャリブレーションが要らない**。モデルAが0.01〜0.3の範囲、モデルBが0.4〜0.9の範囲に予測を出していても、順位に直してから混ぜれば対等に扱えます。生の確率を平均すると、**スケールの大きいモデルの声だけが大きくなる**という不公平が起きます。
2. **外れ値に潰されない**。1件だけ0.9999を出したモデルがあっても、順位に直せばそれは単に「1位」です。生値の平均だとその1件が平均を大きく引っ張ります。
3. **多様性がそのまま利得になる**。相関の低いモデルを混ぜるほど、個々のモデルのノイズが打ち消し合い、順序の精度が上がります。

### この手法の弱点（ここが本題）

- **上限は素材の質で決まる**。混ぜ元より根本的に良くなることはありません。全員が同じ間違いをしていれば、平均しても同じ間違いが残ります。
- **Public LBへの過剰適合**。重みが「Public LBで高かったnotebook」を基準に選ばれているなら、それは**Publicのノイズを学習している**のと同じです。Private LBでは崩れる可能性があります。
- **重みの根拠が無い**。後述のとおり、このnotebookの重み付けは**ファイル名の文字列マッチ**で決まっています。0.35と0.15という数字にも、それを支持する検証結果がありません。
- **再現性が他人依存**。混ぜ元のnotebookが更新・削除されれば、同じスコアは二度と出ません。


## 【解説】コード全体 — Rank-Gauss加重ブレンドの実装

コードは1セルにまとまっているので、ブロックごとに読み解きます。

### 1. 2つの変換関数

```python
def rank01(v):
    return (rankdata(v, method='average') - 0.5) / len(v)

def gauss(v):
    return norm.ppf(np.clip(rank01(v), 1e-7, 1 - 1e-7))
```

- **`rank01`**: 予測値を順位に直し、`(0, 1)` の一様分布にマップします。`method='average'` は同値に平均順位を与える指定（同点を恣意的に並べ替えない）。`- 0.5` を引いているのは、値をちょうど0や1にせず、区間の中央に置くための調整です。
- **`gauss`**: その一様分布を、正規分布の**分位点関数**（`norm.ppf` = 累積分布関数の逆関数）に通して、標準正規分布に変換します。`np.clip` は、0や1を渡すと `ppf` が $\pm\infty$ を返してしまうのを防ぐガードです。

**なぜ一様分布のまま平均しないのか？** ここがこのnotebookの一番の技術的ポイントです。一様分布の順位（percentile）のまま平均すると、**上位1位と2位の差も、中位500位と501位の差も、同じ「1/N」の重み**として扱われます。しかしAUCで効くのは主に**上下の端の並び**です。正規分布に写すと、端に行くほど値の間隔が引き伸ばされる（0.999の分位点は3.09、0.99は2.33、0.5は0）ため、**確信度の高い領域での意見の食い違いが正しく強調されます**。これがRank-Gaussが単純な順位平均より安定して強い理由です。

> 💡 **初心者向け補足**
> - **`scipy.stats.rankdata`**: 配列を順位（1始まり）に変換。`method` で同値の扱いを選べます。
> - **分位点関数 `norm.ppf(q)`**: 「標準正規分布で下側確率が `q` になる値」を返す関数。`ppf(0.5)=0`、`ppf(0.975)≈1.96`。
> - **Rank-Gauss変換**: 元は特徴量の前処理（歪んだ分布をニューラルネットに入れやすくする）として広まった手法。ここでは**出力側**に使っています。

### 2. 提出ファイルの動的収集

```python
paths = glob.glob("/kaggle/input/**/submission.csv", recursive=True)
paths = [p for p in paths if 'sample_submission' not in p.lower()]
```

`glob` の `**` と `recursive=True` で、入力ディレクトリ配下を何階層でも掘って `submission.csv` を集めます。**パスを直書きしないので、Inputを差し替えるだけでブレンド対象を変えられる**のが良い点です。`sample_submission.csv`（全部0.5などのダミー）を除外するのを忘れていないのも重要 — 混ぜてしまうと予測が薄まるだけです。

その後 `merge(..., on='id', how='inner')` で全ファイルを `id` で結合します。`inner` なので、**どれか1つでも欠けているidは丸ごと落ちます**。ここは注意点で、行数が減れば提出は不正になります（実際には全提出が同じidを持つ前提で成立しています）。

### 3. 重み付け ⚠️ 最大の弱点

```python
if 'elite' in col.lower() or 'public' in col.lower() or ... :
    weights.append(0.35)
else:
    weights.append(0.15)
```

**ファイルのパス文字列に特定の単語が含まれるかどうか**で重みを決めています。`elite`, `public`, `psa`, `would`, `ensemble` が入っていれば0.35、それ以外は0.15。最後に合計1になるよう正規化。

これは正直に言って**根拠のないヒューリスティック**です。notebookのタイトルに "elite" と付いているかどうかは、そのモデルの予測性能とは何の関係もありません。作者が「LBで高かったnotebookのタイトルにこれらの単語が含まれていた」ことから逆算しただけで、**モデルの性質ではなくファイル名を見ている**わけです。

**本来どうすべきか**: 手元にOOF（out-of-fold）予測があるなら、**OOF上でAUCを最大化する重みを最適化**するのが正攻法です（`scipy.optimize.minimize` や hill-climbing）。それが無理でも、各予測ペアの**順位相関（Spearman）を見て、相関の低いものに厚く重みを置く**ほうが遥かに筋が通ります。ここは改善余地が最も大きい箇所です。

### 4. 融合と提出

```python
final_gauss += weights[i] * gauss(merged[col].values)
final_preds = rank01(final_gauss)
```

正規化した各予測を重み付きで足し合わせ、最後にもう一度 `rank01` で `(0, 1)` に戻して提出します。**AUCは順序しか見ないので、この最後の変換は本質的には不要**です（スコアは変わりません）。提出値を確率らしい範囲に収めるための整形と考えてよいでしょう。

> 💡 **初心者向け補足**
> - **OOF（out-of-fold）予測**: 交差検証で、各サンプルを「そのサンプルを学習に使っていないモデル」で予測した値。訓練データ全体に対する公平な予測が得られるので、ブレンド重みの最適化に使えます。**これが無いブレンドは、本質的にLBスコアを頼りにした当てずっぽう**になります。
> - **hill climbing（山登り法）**: 候補モデルを1本ずつ「追加して指標が上がるなら採用」と貪欲に足していく重み探索。実装が簡単で過学習しにくく、Kaggleのブレンドで定番です。

---

## 📝 このnotebookからの持ち帰り

1. **指標の性質を知ることが、手法選択そのものになる**。AUCが順序しか見ないと分かっていれば、Rank変換が自然な第一手として出てくる。
2. **Rank-Gaussは単なる順位平均の上位互換**。正規分布に写すことで、確信度の高い端の領域を正しく重く扱える。
3. **重みには根拠を持たせる**。ファイル名マッチではなく、OOFでの最適化か、せめて予測間の相関に基づくべき。
4. **学習ゼロでLB上位に並べてしまうことの意味を考える**。これはPublic LBの脆さの裏返しでもあり、Privateで順位が大きく入れ替わる典型的な要因。**自分の実力を測る指標としては信用してはいけない**。


In [ ]:
#Predicting Smartphone Addiction
# 🚀 [0.97123]
# If this notebook helps you, please UPVOTE! ⬆️

import pandas as pd
import numpy as np
import glob
from scipy.stats import rankdata, norm
import warnings
warnings.filterwarnings('ignore')

def rank01(v):
    return (rankdata(v, method='average') - 0.5) / len(v)

def gauss(v):
    return norm.ppf(np.clip(rank01(v), 1e-7, 1 - 1e-7))

print("🔍 Loading top public submissions...")

# Find all submission paths dynamically
paths = glob.glob("/kaggle/input/**/submission.csv", recursive=True)
paths = [p for p in paths if 'sample_submission' not in p.lower()]

if len(paths) < 5:
    print(f"Warning: Expected 5 submissions, found {len(paths)}.")

dfs = []
for p in paths:
    df = pd.read_csv(p)
    dfs.append(df[['id', 'addicted_label']].rename(columns={'addicted_label': p}))

# Merge all submissions
merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on='id', how='inner')

# Optimized weights (rewarding generative & diverse models)
weights = []
for col in merged.columns:
    if col == 'id': continue
    if 'elite' in col.lower() or 'public' in col.lower() or 'psa' in col.lower() or 'would' in col.lower() or 'ensemble' in col.lower():
        weights.append(0.35)
    else:
        weights.append(0.15)

weights = np.array(weights)
weights = weights / weights.sum() # Normalize to 1.0

print(f"✅ Blending {len(weights)} models with Rank-Gauss scaling...")

# Apply Rank-Gauss and weighted average
final_gauss = np.zeros(len(merged))
for i, col in enumerate([c for c in merged.columns if c != 'id']):
    final_gauss += weights[i] * gauss(merged[col].values)

# Convert back to ranks for submission
final_preds = rank01(final_gauss)

# Save
sub_path = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)[0]
sub = pd.read_csv(sub_path)
sub['addicted_label'] = final_preds
sub.to_csv('submission.csv', index=False)

print(f"🏆 Blend complete! Saved submission.csv (Target: 0.97123)")
